<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/QA/NLPAPP_KnowledgeBase_based_Factoid_Multihop_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip -q install sentence-transformers scikit-learn pandas numpy

In [ ]:
import re
import numpy as np
import pandas as pd

from collections import defaultdict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 0. Knowledge Base

In [ ]:
# Hardcoded Knowledge Graph: (subject, relation, object)
knowledge_graph = [("A", "works_in", "Engineering"),("B", "works_in", "Finance"),("C", "works_in", "Engineering"),("Engineering", "director", "R"),("Finance", "director", "E"),
    ("A", "manager", "J"), ("B", "manager", "S"),("J", "reports_to", "R"),("S", "reports_to", "E"), ("R", "role", "CTO"), ("E", "role", "CFO"),
]
qa_test = [
    {"question": "Who manages A?","ground_truth": "J"},
    {"question": "Who is the director of the department where C works?","ground_truth": "R"},
    {"question": "Who does B 's manager report to?","ground_truth": "E"},
    {"question": "What role does R have?","ground_truth": "CTO"},
    {"question": "Which department does C work in?","ground_truth": "Engineering"}
]


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_graph(triples):
    G = nx.DiGraph()
    for subj, rel, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj,relation=rel)
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G,seed=42,k=2)
    nx.draw_networkx_nodes(G,pos,node_color="skyblue",node_size=3500,edgecolors="black")
    nx.draw_networkx_labels(G,pos,font_size=10,font_weight="bold")
    nx.draw_networkx_edges(G,pos,arrowstyle="->",arrowsize=20,edge_color="gray",width=2)
    edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G,pos,edge_labels=edge_labels,font_color="red",font_size=9)
    plt.title("Knowledge Graph Visualization",fontsize=16,fontweight="bold")
    plt.axis("off")
    plt.show()

In [ ]:
visualize_graph(knowledge_graph)

## 1. Query Processor

In [ ]:
#This code is not used. Added for completion and to provide a placeholder for students to implement GEC if required!
class QueryProcessor:
    def process(self, question):
        question = question.lower()
        return question

## 2. Knowledge Graph Processor(Query Logic)

In [ ]:
class KnowledgeGraph:
    def __init__(self, triples):
        self.graph = defaultdict(list)
        self.entities = set()
        self.relations = set()
        for subj, rel, obj in triples:
            self.graph[subj].append((rel, obj))
            self.entities.add(subj)
            self.entities.add(obj)
            self.relations.add(rel)

    def get_relation(self, entity, relation):
        for rel, obj in self.graph[entity]:
            if rel == relation:
                return obj
        return None
    def get_all_relations(self, entity):
        return self.graph[entity]

### Semantic Parsing
* Since NER is yet to discussed , here simple relation pattern based parsing is used. This entire pipeline can be replaced by LLM

In [ ]:
class SemanticParser:
    def __init__(self, kg):
        self.kg = kg
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.relation_phrases = {"manager": ["manager","manages","supervisor","boss"],
            "works_in": ["works in","department","team"],"director": ["director","head","leader"],
            "reports_to": ["reports to","reporting manager"],"role": ["role","designation","position"]
        }
        self.relation_embeddings = {}
        for relation, phrases in self.relation_phrases.items():
            self.relation_embeddings[relation] = self.embedder.encode(phrases)

    def extract_entity(self, question):
        for entity in self.kg.entities:
            if entity.lower() in question.lower():
                return entity
        return None

    def detect_relation(self, question):
        q_emb = self.embedder.encode([question])[0]
        best_relation = None
        best_score = -1
        for relation, emb_list in self.relation_embeddings.items():
            for emb in emb_list:
                score = cosine_similarity([q_emb],[emb])[0][0]
                if score > best_score:
                    best_score = score
                    best_relation = relation
        return best_relation

    def detect_multihop(self, question):
        question = question.lower()
        if "director of the department" in question:
            return ["works_in","director"]
        if "manager report to" in question:
            return ["manager", "reports_to"]
        return None

In [ ]:
kgT = KnowledgeGraph(knowledge_graph)
parserT = SemanticParser(kgT)
questionT = qa_test[2]["question"]
entityT = parserT.extract_entity(questionT)
relationT = parserT.detect_relation(questionT)
multihopT = parserT.detect_multihop(questionT)
print("Sample Query:", questionT)
print("\tDetected Entity:", entityT)
print("\tDetected Relation:", relationT)
print("\tMulti-hop Chain:", multihopT)


## 3. Candidate Retriever (Reasoning Engine)

In [ ]:
class Reasoner:
    def __init__(self, kg):
        self.kg = kg

    def single_hop(self, entity, relation):
        return self.kg.get_relation(
            entity,
            relation
        )

    def multi_hop(self, entity, relations):
        current = entity
        reasoning_path = [entity]
        for rel in relations:
            current = self.kg.get_relation(current,rel)
            reasoning_path.append(rel)
            reasoning_path.append(current)
            if current is None:
                return None, reasoning_path
        return current, reasoning_path

In [ ]:
reasonerT = Reasoner(kgT)

print("Sample Query:", questionT)
print("\tDetected Entity:", entityT)
print("\tDetected Relation:", relationT)
print("\tMulti-hop Chain is it there ?:", multihopT)

if multihopT:
  answerMT, reasoning_pathMT = reasonerT.multi_hop(entityT,multihopT)
  reasoning_pathMT = [entityT,relationT,answerMT]
  print("Reasoning Path if any in multihop for the answer:",answerMT)
  print(reasoning_pathMT)

answerST = reasonerT.single_hop(entityT,relationT)
print("Answer if found in singlehop for the query is :",answerST)


## 3. Answer Generator (Post Processing)

In [ ]:
#Learners may use this section to code automated modification/rewrite/reordering/filtering of candidate answers before final answers gets curated
class PostProcessor:
    def curate(self, answer):
        if answer is None:
            return "No answer found"
        return answer.strip()

## 4. Evaluator

In [ ]:
class Evaluator:
    def exact_match(self, pred, gt):
        return int(pred.lower() == gt.lower())

    def path_accuracy(self, path):
        return int(None not in path)

    def aggregate(self, em, path_acc):
        return (em + path_acc) / 2

In [ ]:
evaluatorT = Evaluator()

gtT = qa_test[1]["ground_truth"]
if answerST:
  final_answer=answerST
else:
  final_answer=answerMT

emT = evaluatorT.exact_match(final_answer,gtT)
path_accT = evaluatorT.path_accuracy(reasoning_pathMT)
final_scoreT = evaluatorT.aggregate(emT,path_accT)

print("Ground Truth:", gtT)
print("Predicted Answer:", final_answer)
print("Exact Match:", emT)
print("Path Accuracy:", path_accT)
print("Final Score:",round(final_scoreT, 3))



##

## Pipeline

In [ ]:
print("=" * 60)
print("SEMANTIC KNOWLEDGE-BASED QA")
print("=" * 60)

kg = KnowledgeGraph(knowledge_graph)
parser = SemanticParser(kg)
reasoner = Reasoner(kg)
postprocessor = PostProcessor()
evaluator = Evaluator()

all_scores = []
results = []

for sample in qa_test:
    print("\n" + "=" * 60)
    question = sample["question"]
    gt = sample["ground_truth"]
    print("\nQUESTION:")
    print(question)

    print("\n[PHASE 1] SEMANTIC PARSING")
    entity = parser.extract_entity(question)
    relation = parser.detect_relation(question)
    multihop = parser.detect_multihop(question)
    print("Detected Entity:", entity)
    print("Detected Relation:", relation)
    print("Multi-hop Chain:", multihop)
    print("\n[PHASE 2] GRAPH PROCESSING. NO specific indexing is needed in this toy problem!")
    print("\n[PHASE 3] REASONING")
    if multihop:
        answer, reasoning_path = reasoner.multi_hop(entity,multihop)
    else:
        answer = reasoner.single_hop(entity,relation)
        reasoning_path = [entity,relation,answer]
    print("Reasoning Path:")
    print(reasoning_path)
    print("\n[PHASE 4] ANSWER GENERATION")
    candidate = answer
    print("\n[PHASE 5] POSTPROCESSING")
    final_answer = postprocessor.curate(candidate)
    print("Final Answer:", final_answer)
    print("\n[PHASE 6] EVALUATION")
    em = evaluator.exact_match(final_answer,gt)
    path_acc = evaluator.path_accuracy(reasoning_path)
    final_score = evaluator.aggregate(em,path_acc)
    all_scores.append(final_score)
    print("Ground Truth:", gt)
    print("Exact Match:", em)
    print("Path Accuracy:", path_acc)
    print("Final Score:",round(final_score, 3))
    results.append({"Question": question,"Predicted": final_answer,"Ground Truth": gt,"Reasoning Path": str(reasoning_path),
        "Exact Match": em,"Path Accuracy": path_acc,"Final Score": round(final_score, 3)})

print("\n" + "=" * 60)
print("FINAL AGGREGATE RESULTS")
print("=" * 60)
avg_score = sum(all_scores) / len(all_scores)
print("\nAverage Overall Score:",
      round(avg_score, 3))
df = pd.DataFrame(results)
print("\nDETAILED RESULTS")
display(df)

In [ ]:
#If you wish to modularize and make each of above function resuable in your local repository, create seperate file . Then you may import them in codes eg., below:
#from data import documents, qa_test
#from embedder import Embedder
#from retriever import SemanticRetriever
#from answer_generator import AnswerGenerator
#from evaluator import Evaluator